# Lesson 3b — Preconditioners, IC(0), and variable-coefficient solvers

Prereq: Lesson 3 (GFM Poisson).

The GFM pressure solve in Lesson 3 gave you a symmetric positive-definite (SPD) matrix. In 1-D with `N = 80` unknowns, SciPy's direct `spsolve` handled it instantly. In 3-D with `N = 256³ ≈ 1.7 · 10⁷` unknowns, direct solves are off the table. You need an iterative solver — **Conjugate Gradient** (CG) is the natural choice for SPD systems — and for CG to converge fast enough to be useful, you need a **preconditioner**.

## 1. What is a preconditioner?

An iterative solver for `A x = b` generates a sequence of approximate solutions `x_k → x*`. Each step costs one sparse matrix-vector product `A v` plus a bit of linear algebra. The question is how many steps it takes.

For CG, the answer depends on the **condition number** `κ(A) = λ_max / λ_min` (ratio of largest to smallest eigenvalue). The error bound is

$$\|x_k - x^*\|_A \le 2\left(\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}\right)^k \|x_0 - x^*\|_A$$

so iteration count scales like `O(√κ)`. For a 5-point 2-D Laplacian on an `N × N` grid, `κ = O(N²)`, so CG takes `O(N)` iterations — each iteration has `O(N²)` work — for a total of `O(N³)`. Workable but slow on fine grids.

A **preconditioner** `M` is a matrix that is:

1. A good approximation of `A` (so `M⁻¹A` has a smaller condition number than `A`),
2. Cheap to apply (you need to compute `M⁻¹r` in every CG iteration).

Instead of solving `A x = b`, you solve the equivalent preconditioned system. For SPD `A` and SPD `M`, the preconditioned-CG (PCG) iteration count scales like `O(√κ(M⁻¹A))`, which can be dramatically smaller than `O(√κ(A))`.

The tradeoff: **each PCG iteration is more expensive** (one `M⁻¹r` solve added), but you do fewer of them. A good preconditioner gives a net win by orders of magnitude.

## 2. Common preconditioners, from cheap to powerful

| `M` | Quality | Apply cost | Notes |
|-----|---------|-----------:|-------|
| `I` (identity) | none | free | vanilla CG |
| `diag(A)` (**Jacobi**) | weak | `O(N)` | one division per unknown |
| Gauss–Seidel / SOR | moderate | `O(nnz)` | triangular sweep |
| **IC(0)** — incomplete Cholesky, zero fill | strong | `O(nnz)` setup, `O(nnz)` apply | what this lesson is about |
| Multigrid | very strong | `O(N)` apply | `O(N)` total solver, hardest to implement |
| Algebraic multigrid | very strong | `O(N)` apply | multigrid without the geometry |

For the fire pressure solve, the practical choice is **IC(0)-preconditioned CG** (ICPCG) on coarse-to-medium grids, or geometric multigrid on very fine grids. Nguyen/Fedkiw/Jensen 2002 use ICPCG.

## 3. Incomplete Cholesky with zero fill: IC(0)

For SPD `A`, the exact Cholesky factorization `A = L Lᵀ` has `L` lower-triangular. Computing `L` costs `O(N³)` dense or `O(nnz · bandwidth)` sparse — and crucially, `L` can be **much denser than `A`**. The new nonzeros are called **fill-in**. For a 5-point 2-D Laplacian on an `N×N` grid, `A` has `5N²` nonzeros; `L` has `O(N³/2)`. That's unacceptable at large `N`.

**IC(0)** is the obvious cheat: compute `L` *as if* you were doing Cholesky, but **skip any entry whose position is not already a nonzero of `A`**. The `(0)` means "zero fill" — no new nonzeros introduced. The result `L̃` satisfies `L̃ L̃ᵀ ≈ A` (not equal — you threw out the fill-in) but has exactly the same sparsity pattern as the lower triangle of `A`.

### Algorithm (in pseudo-code)

```
for i = 0 .. N-1:
    # diagonal: L_ii = sqrt(A_ii − Σ_{k<i} L_ik²)   (summed only over k where L_ik ≠ 0)
    s = A[i,i]
    for k < i where (i,k) ∈ pattern(A):
        s -= L[i,k]²
    L[i,i] = sqrt(s)

    # column i below diagonal: L_ji = (A_ji − Σ_{k<i} L_jk L_ik) / L_ii
    for j > i where (j,i) ∈ pattern(A):
        s = A[j,i]
        for k < i where (j,k) ∈ pattern(A) AND (i,k) ∈ pattern(A):
            s -= L[j,k] * L[i,k]
        L[j,i] = s / L[i,i]
```

Apply `M⁻¹ = (L̃ L̃ᵀ)⁻¹` via two triangular solves per iteration: `y = L̃⁻¹ r`, then `z = L̃⁻ᵀ y`. Both are `O(nnz)`.

### Why it works so well

Fill-in is dominated by entries that are small in magnitude (heuristically — they come from long-range interactions). Dropping them preserves most of the factorization's accuracy. For Poisson-like problems, IC(0) typically gives `κ(M⁻¹A) = O(N)` instead of `O(N²)` — so `O(√N)` iterations instead of `O(N)`, a win that grows with grid size.

### Warning: IC(0) can fail

`sqrt(s)` can encounter `s ≤ 0` if `A` is not M-matrix-like or is poorly scaled. Robust fixes: add a small diagonal shift `A + α I` before factoring (MIC — **modified** IC), or use IC(k) with `k > 0` fill levels for more robustness. For the fire pressure solve, the matrix is a discrete Laplacian with `1/ρ` coefficients — always SPD and well-behaved — so IC(0) works out of the box.

## 4. What is a variable-coefficient solver?

The fire pressure Poisson equation is

$$-\nabla\!\cdot\!\left(\tfrac{1}{\rho(\mathbf{x})}\nabla p\right) = f$$

When `ρ` is a constant, this reduces to `-Δp = ρ f` — a **constant-coefficient** Laplacian that you can solve with an FFT in `O(N log N)`. But in fire, `ρ = ρ_f` in fuel and `ρ = ρ_h` in hot gas, **different by an 8× factor**. FFT methods no longer apply.

A **variable-coefficient solver** is just any sparse linear solver that handles a discrete operator whose off-diagonal entries vary cell-by-cell. In our case that's:

- 5-point / 7-point stencil (same shape as constant-coefficient)
- Coefficients `1/ρ̂` on each face, different per face (Lesson 3, Section 5)
- Still symmetric positive-definite

Nothing about the stencil structure changes; what changes is the *values* of the matrix entries and consequently the condition number. With `ρ_f/ρ_h = 8`, `κ(A)` is roughly `8×` worse than the constant-ρ case, so preconditioning matters more, not less.

**Key point**: CG and IC(0) **don't care** that the coefficients vary. They only need SPD structure. You assemble the variable-coefficient matrix once, factor it with IC(0) once, and iterate. The variable-coefficient nature only manifests in:

1. How you *assemble* the matrix (Lesson 3 §5 — use the length-weighted `ρ̂`).
2. The condition number you end up with (worse ratio → more iterations → preconditioning more critical).
3. You can't use FFT-based direct solvers; need iterative + preconditioner.

The fire solver's pressure step is the archetypal variable-coefficient elliptic solve.

## 5. Interactive: CG vs PCG-Jacobi vs PCG-IC(0)

Build a 2-D variable-ρ Laplacian with a sharp density contrast, and compare the three iterative solvers. The metric is iteration count to reach a given residual tolerance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csr_matrix, diags
from scipy.sparse.linalg import cg, LinearOperator, spsolve_triangular

def build_2d_variable_rho_laplacian(N, rho):
    """Build -∇·(1/ρ ∇p) on N×N cell-centered grid with Dirichlet BCs.
    rho is shape (N, N). Uses harmonic average at faces (no front, no jump)."""
    h = 1.0 / N
    n = N * N
    A = lil_matrix((n, n))
    for j in range(N):
        for i in range(N):
            k = j*N + i
            # Face conductances 1/ρ̂, using harmonic average for continuous ρ
            def face_coef(ra, rb):
                return 2.0 / (ra + rb)      # harmonic; equals 1/ρ for equal ρ
            # Right face
            if i < N-1:
                c = face_coef(rho[j,i], rho[j,i+1]) / h**2
                A[k,k] += c; A[k, k+1] -= c
            else:
                A[k,k] += 2.0 / rho[j,i] / h**2
            # Left face
            if i > 0:
                c = face_coef(rho[j,i], rho[j,i-1]) / h**2
                A[k,k] += c; A[k, k-1] -= c
            else:
                A[k,k] += 2.0 / rho[j,i] / h**2
            # Up face
            if j < N-1:
                c = face_coef(rho[j,i], rho[j+1,i]) / h**2
                A[k,k] += c; A[k, k+N] -= c
            else:
                A[k,k] += 2.0 / rho[j,i] / h**2
            # Down face
            if j > 0:
                c = face_coef(rho[j,i], rho[j-1,i]) / h**2
                A[k,k] += c; A[k, k-N] -= c
            else:
                A[k,k] += 2.0 / rho[j,i] / h**2
    return A.tocsr(), h

In [ ]:
def incomplete_cholesky_0(A_csr):
    """IC(0): L such that L Lᵀ ≈ A, with pattern(L) = lower-tri pattern of A."""
    n = A_csr.shape[0]
    A = A_csr.tolil()
    # Seed L with the lower triangle of A (includes diagonal).
    L_rows = [dict() for _ in range(n)]
    for i in range(n):
        for j, v in zip(A.rows[i], A.data[i]):
            if j <= i:
                L_rows[i][j] = v

    for i in range(n):
        s = L_rows[i][i]
        for k in L_rows[i]:
            if k < i:
                s -= L_rows[i][k]**2
        if s <= 0:
            raise ValueError(f'IC(0) breakdown at row {i}, s={s}')
        L_rows[i][i] = np.sqrt(s)
        # Update column i below diagonal for rows j where (j,i) ∈ pattern
        for j in range(i+1, n):
            if i in L_rows[j]:
                s = L_rows[j][i]
                for k in L_rows[j]:
                    if k < i and k in L_rows[i]:
                        s -= L_rows[j][k] * L_rows[i][k]
                L_rows[j][i] = s / L_rows[i][i]

    rows, cols, data = [], [], []
    for i in range(n):
        for j, v in L_rows[i].items():
            rows.append(i); cols.append(j); data.append(v)
    return csr_matrix((data, (rows, cols)), shape=(n, n))


def ic0_solve(L, r):
    """Apply (L Lᵀ)⁻¹ via two triangular solves."""
    y = spsolve_triangular(L, r, lower=True)
    return spsolve_triangular(L.T.tocsr(), y, lower=False)

In [ ]:
# Build a problem with a sharp ρ contrast (mimicking fuel vs hot)
N = 48
X, Y = np.meshgrid((np.arange(N)+0.5)/N, (np.arange(N)+0.5)/N, indexing='xy')
rho = np.where(((X - 0.5)**2 + (Y - 0.5)**2) < 0.15**2, 1.0, 0.125)   # dense disc in middle

A, h = build_2d_variable_rho_laplacian(N, rho)
n = N * N

# Random but consistent RHS
np.random.seed(0)
b = np.random.randn(n)
b -= b.mean()   # make compatible with Neumann-like structure (not strictly needed here)

# Factor IC(0)
L = incomplete_cholesky_0(A)
print(f'Matrix A: {A.nnz} nonzeros; IC(0) factor L: {L.nnz} nonzeros (same lower-tri pattern)')

# Preconditioners as LinearOperators
D_inv = 1.0 / A.diagonal()
M_jacobi = LinearOperator((n, n), matvec=lambda r: D_inv * r)
M_ic0    = LinearOperator((n, n), matvec=lambda r: ic0_solve(L, r))

# Run CG with each; log residual per iteration
def run(M, label):
    history = []
    def cb(xk):
        history.append(np.linalg.norm(b - A @ xk))
    x, info = cg(A, b, rtol=1e-8, atol=0, maxiter=500, M=M, callback=cb)
    return np.array(history), info

hist_none,   _ = run(None,     'CG')
hist_jacobi, _ = run(M_jacobi, 'PCG-Jacobi')
hist_ic0,    _ = run(M_ic0,    'PCG-IC(0)')

plt.figure(figsize=(8, 5))
plt.semilogy(hist_none,   label=f'CG (no preconditioner), {len(hist_none)} iters')
plt.semilogy(hist_jacobi, label=f'PCG-Jacobi, {len(hist_jacobi)} iters')
plt.semilogy(hist_ic0,    label=f'PCG-IC(0), {len(hist_ic0)} iters')
plt.xlabel('iteration'); plt.ylabel('residual ‖b − A x‖')
plt.title(f'CG convergence on variable-ρ Laplacian ({N}×{N}, ρ contrast 8:1)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### What you should see

- **Vanilla CG** grinds through many iterations — hundreds on this 48² grid.
- **PCG-Jacobi** helps a bit (maybe 2× fewer iterations) — it handles the diagonal scaling from the ρ contrast, but not the spatial structure.
- **PCG-IC(0)** converges in a small fraction of the iteration count — on this problem typically 5–10× fewer iterations than vanilla CG.

Iteration ratio grows with `N`. Rerun with `N = 96` and the IC(0) advantage roughly doubles. That's the whole reason this preconditioner is worth the setup cost.

## 6. Putting it into the fire solver

Per frame of the fire loop (Lesson 4, step 8):

1. **Assemble** the variable-coefficient Laplacian `A` using Lesson 3 §5's face coefficients and GFM jumps on the RHS.
2. **Factor** it with IC(0). Reuse the factor across frames if the matrix structure is stable (it isn't in fire — the level set moves — but you can reuse within a substep).
3. **PCG-iterate** until `‖r‖ < tol` (typically `10⁻⁵` to `10⁻⁷`).
4. **Apply** the pressure correction `u ← u* − (Δt/ρ) ∇p`.

On fine grids (`256³` and up), switch to geometric multigrid or AMG. Both are well-studied; for a learning implementation, IC(0) + PCG is enough to handle `64³` – `128³` problems in real seconds per frame.

## Summary

- **Preconditioner**: a matrix `M ≈ A` that's cheap to invert. Solving `M⁻¹ A x = M⁻¹ b` iteratively converges faster than solving `A x = b` directly.
- **IC(0)**: drop-in-place Cholesky; factor `L L^T ≈ A` with `pattern(L) = lower-tri(A)`. Memory cost equals `A` itself; apply via two triangular solves.
- **Variable-coefficient solver**: any sparse linear solver applied to a discrete operator whose coefficients vary cell-by-cell. Structurally identical to constant-coefficient; only the numbers change. The fire pressure solve is one.
- **Why it matters for fire**: density contrast `ρ_f/ρ_h ≈ 8` worsens the condition number; IC(0)+PCG recovers the lost iterations, making the pressure solve cheap enough to run per frame.